# PWV03b : Two-Point Temporal Correlation — noise-floor correction

## Overview

This notebook corrects the estimator from **PWV03** so that $C(\Delta t \to 0) \approx 1$.

### Why C(0) < 1 in PWV03

Each PWV measurement has **two independent noise contributions** beyond the true
atmospheric signal:

$$
\mathrm{PWV}_i^{\rm obs} = \mathrm{PWV}_i^{\rm atm}
  + \epsilon_i^{\rm stat} + \epsilon_i^{\rm rep},
$$

* $\sigma_{\rm stat,i}$ = statistical error from photon noise in the CCD pixels,
  **as returned by Spectractor** in `PWV_err`. This varies per exposure.
* $\sigma_{\rm rep} \approx 0.12\,\mathrm{mm}$ = **instrumental repeatability**:
  intrinsic scatter between back-to-back measurements of the same atmospheric
  state (e.g. from PSF variations, flat-field residuals, sky subtraction). It
  is **NOT captured by `PWV_err`**.

The total intra-night variance is therefore:
$$
\sigma^2_{\rm obs} = \sigma^2_{\rm phys}
  + \underbrace{\bar{\sigma}^2_{\rm stat} + \sigma^2_{\rm rep}}_{\sigma^2_{\rm noise}},
$$
and for **two distinct measurements** $i \ne j$ the noise terms are independent,
so the cross-product expectation at zero physical lag is:
$$
\langle\delta\mathrm{PWV}_i\,\delta\mathrm{PWV}_j\rangle_{\Delta t\to 0}
  = \sigma^2_{\rm phys}
  = \sigma^2_{\rm obs} - \sigma^2_{\rm noise}.
$$
Normalising by $\sigma^2_{\rm obs}$ (as in PWV03) gives:
$$
C(0) = \frac{\sigma^2_{\rm phys}}{\sigma^2_{\rm obs}} = 1 - \frac{\sigma^2_{\rm noise}}{\sigma^2_{\rm obs}} < 1.
$$
Because PWV03 only subtracts $\bar{\sigma}^2_{\rm stat}$ (from `PWV_err`), the
repeatability term $\sigma^2_{\rm rep}$ is missed, and the correction is insufficient.

### Correction strategy

Normalise by $\sigma^2_{\rm phys} = \sigma^2_{\rm obs} - \sigma^2_{\rm noise}$ where
$$
\sigma^2_{\rm noise} = \bar{\sigma}^2_{\rm stat} + \sigma^2_{\rm rep}.
$$
Two methods are provided to obtain $\sigma_{\rm rep}$:

* **Method A** — fixed prior: $\sigma_{\rm rep} = 0.12\,\mathrm{mm}$ (from repeatability
  measurements done separately).
* **Method B** — empirical via the Structure Function:
  $\mathrm{SF}(\Delta t) = \langle(\delta\mathrm{PWV}_i - \delta\mathrm{PWV}_j)^2\rangle$.
  At zero lag, $\mathrm{SF}(0) = 2\,\sigma^2_{\rm noise}$, so extrapolating the
  SF to $\Delta t=0$ yields $\sigma_{\rm rep}$ from the data directly.

**Author :** Sylvie Dagoret-Campagne — IJCLab/IN2P3/CNRS  
**Created :** 2026-03-19  
**Modified :** repeatability-aware noise correction  
**Kernel :** conda_py313


Le vrai problème : PWV_err (le sigma Spectractor) ne capture que les fluctuations statistiques photoniques du CCD. Mais il existe une répétabilité intrinsèque $\sigma_{\rm rep} \approx 0.12$ mm qui est la vraie dispersion minimale observable entre deux mesures successives du même état atmosphérique — et c'est elle qui "bouffe" le numérateur de C(Δt→0).

Ce qu'on observe réellement :
$\langle\delta\mathrm{PWV}_i\,\delta\mathrm{PWV}_j\rangle_{\Delta t\to 0} = \sigma^2_{\rm phys} \neq \sigma^2_{\rm obs}⟨$
parce que $\sigma^2_{\rm obs} = \sigma^2_{\rm phys} + \sigma^2_{\rm rep}$
avec $\sigma_{\rm rep}$ non mesurée directement par les erreurs fit.
La bonne approche sans soustraction : utiliser l'estimateur de la limite $\Delta t \to 0$
directement depuis les données — c'est-à-dire extrapoler $C(\Delta t)$
vers $\Delta t = 0$ en fittant le premier régime, et
renormaliser par la valeur extrapolée, ou bien calculer $\sigma^2_{\rm rep}$ empiriquement comme la demi-variance des différences entre paires consécutives très proches.

Voici ma proposition concrète :Excellente clarification ! Le vrai problème est fondamentalement différent — et bien plus intéressant physiquement.
Le bon modèle : chaque mesure PWV a deux sources de bruit indépendantes :
$\mathrm{PWV}_i^{\rm obs} = \mathrm{PWV}_i^{\rm atm} + \epsilon_i^{\rm stat} + \epsilon_i^{\rm rep}$
où $\sigma_{\rm stat,i}$ est fourni par Spectractor (bruit photonique), et $\sigma_{\rm rep} \approx 0.12$ mm est la répétabilité instrumentale 
non mesurée par les erreurs fit, donc pas dans PWV_err.

Le produit croisé pour deux mesures distinctes donne :
$\langle\delta\mathrm{PWV}_i\,\delta\mathrm{PWV}_j\rangle_{\Delta t\to 0} = \sigma^2_{\rm phys} \quad (\text{termes croisés nuls par indépendance})$

mais la variance totale est :

$\sigma^2_{\rm obs} = \sigma^2_{\rm phys} + \sigma^2_{\rm stat} + \sigma^2_{\rm rep}$
La correction sans soustraction consiste à estimer $\sigma^2_{\rm rep}$ directement depuis les données —
via la structure-function (demi-variance des différences de paires très proches) — puis à renormaliser proprement. Voici le diagramme du raisonnement :

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.resetwarnings()
warnings.simplefilter('ignore')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from astropy.time import Time
from scipy.optimize import curve_fit

%matplotlib inline

mpl.rcParams.update({
    'figure.dpi'     : 150,
    'font.family'    : 'serif',
    'font.size'      : 11,
    'axes.labelsize' : 13,
    'axes.titlesize' : 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.grid'      : True,
    'grid.alpha'     : 0.35,
    'axes.linewidth' : 1.1,
})

In [ ]:
from PWV00_parameters import (
    version_run, tag, extractedfilesdict,
    PWV_FILTER_LIST, FLAG_PWVFILTERS,
    DumpConfig
)
from mysitcom.auxtel.qualitycuts import ParameterCutSelection, ParameterCutTools
from mysitcom.auxtel.pwv import normalize_column_data_bytarget_byfilter

DumpConfig()

In [ ]:
pathfigs = "figs_PWV03corr"
prefix   = "pwv03bcorr"
os.makedirs(pathfigs, exist_ok=True)
figtype  = ".pdf"

In [ ]:
FLAG_WITHCOLLIMATOR = False
datetime_WITHCOLLIMATOR = pd.to_datetime("2023-09-30 00:00:00.0+0000")

---
## 1. Load data & build Time column

In [ ]:
atmfilename   = extractedfilesdict[version_run]
inputfilename = atmfilename.split("/")[-1]
print(f"Loading: {atmfilename}")

if "parquet" in inputfilename:
    df_spec = pd.read_parquet(atmfilename)
else:
    specdata = np.load(atmfilename, allow_pickle=True)
    df_spec  = pd.DataFrame(specdata)

df_spec["Time"] = pd.to_datetime(df_spec["DATE-OBS"], utc=True)
df_spec = df_spec.sort_values("Time", ascending=True).reset_index(drop=True)

if FLAG_WITHCOLLIMATOR:
    df_spec = df_spec[df_spec["Time"] > datetime_WITHCOLLIMATOR].copy()

print(f"Shape: {df_spec.shape}")
print(f"Date range: {df_spec['Time'].min().date()}  ->  {df_spec['Time'].max().date()}")

In [ ]:
## Re-compute nightObs from Time (id is buggy in Corentin's files)

In [ ]:
df_spec["nightObs"] = df_spec.apply(lambda x: x['id']//100_000, axis=1)
df_spec["seq_num"]  = df_spec["id"] % 100_000
df_spec["night_from_time_utc"] = (
    df_spec["Time"].dt.strftime("%Y%m%d").astype(int)
)

mismatch = df_spec[
    df_spec["night_from_time_utc"] != df_spec["nightObs"]
]

len(mismatch), len(df_spec)

In [ ]:
FLAG_CORRECT_NIGHTOBS_VARIABLES = True

if FLAG_CORRECT_NIGHTOBS_VARIABLES and "run2026_v02" in version_run:
    print("COMPUTE NIGHTOBS FROM Time") 
    df_spec["Time_local"] = df_spec["Time"].dt.tz_convert("America/Santiago")
    df_spec["nightObs_corr"] = (
        (df_spec["Time_local"] - pd.Timedelta(hours=12))
        .dt.strftime("%Y%m%d")
        .astype(int)
    )
if FLAG_CORRECT_NIGHTOBS_VARIABLES and "run2026_v02" in version_run:
    span = (
        df_spec.groupby("nightObs_corr")["Time"].max()
        - df_spec.groupby("nightObs_corr")["Time"].min()
    )
    print(span.max())
if FLAG_CORRECT_NIGHTOBS_VARIABLES and "run2026_v02" in version_run:
    df_spec["nightObs"] = df_spec["nightObs_corr"]

In [ ]:
if "run2026_v02" in version_run:
    df_spec["chi2_ram"] = df_spec["CHI2_FIT"]
    df_spec["chi2_rum"] = df_spec["CHI2_FIT"]

if FLAG_PWVFILTERS:
    df_spec = df_spec[df_spec["FILTER"].isin(PWV_FILTER_LIST)].copy()

pwv_cols = ["PWV [mm]_ram", "PWV [mm]_rum", "PWV [mm]_err_ram", "PWV [mm]_err_rum"]
df_spec.dropna(subset=pwv_cols, inplace=True)

print(f"After filter selection: {len(df_spec)} rows")

In [ ]:
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="CHI2_FIT", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_ram", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_rum", ext="norm")

---
## 2. Quality cuts

In [ ]:
FLAG_LOOSE_CUTS = True
FLAG_TIGHT_CUTS = False

In [ ]:
pathdata      = "data_PWV01seas"
if FLAG_LOOSE_CUTS:
    filename_cuts = f"{pathdata}/cuts_loose_finaldecision.json"
    cutstype_name = "loose-cuts"
elif FLAG_TIGHT_CUTS:
    filename_cuts = f"{pathdata}/cuts_tight_finaldecision.json"
    cutstype_name = "tight-cuts"
else:
    filename_cuts = f"{pathdata}/cuts_finaldecision.json"
    cutstype_name = "standard-cuts"

cuts           = ParameterCutTools.load_cuts_json(filename_cuts)
list_of_params = list(cuts.keys())

selector    = ParameterCutSelection(df_spec, params=list_of_params, id_col="id")
flags       = selector.apply_cuts(cuts)
df_selected = df_spec.merge(flags, on="id")
df_keep     = df_selected[df_selected["pass_all_cuts"]].copy()

print(f"Cuts ({cutstype_name}): {len(df_keep)} / {len(df_spec)} kept")

---
## 3. Prepare time series & variance budget

### Noise model

The total noise floor is:
$$
\sigma^2_{\rm noise} = \bar{\sigma}^2_{\rm stat} + \sigma^2_{\rm rep},
$$
where $\bar{\sigma}^2_{\rm stat} = \langle\sigma_{\rm PWV,err}^2\rangle$ is the
mean photon-noise variance (from `PWV_err`) and $\sigma_{\rm rep}$ is the
instrumental repeatability.

### Two methods for σ_rep

**Method A — fixed prior:**  
$\sigma_{\rm rep}$ is set to a fixed value measured independently.
Set `SIGMA_REP_PRIOR` below.

**Method B — structure function:**  
The structure function $\mathrm{SF}(\Delta t) = \langle(\delta\mathrm{PWV}_i - \delta\mathrm{PWV}_j)^2\rangle$
satisfies:
$$
\mathrm{SF}(\Delta t) = 2\,\sigma^2_{\rm obs}\bigl[1 - C(\Delta t)\bigr]
\xrightarrow{\Delta t\to 0} 2\,\sigma^2_{\rm noise}.
$$
We compute SF in the same log-bins as C, then fit a power law
$\mathrm{SF}(\Delta t) = A\,\Delta t^\beta + \mathrm{SF}_0$ to extract
$\mathrm{SF}_0 = 2\,\sigma^2_{\rm noise}$ by extrapolation to $\Delta t=0$.

Both methods are computed; the active denominator is selected via `USE_METHOD`.


In [ ]:
# ── Choice of method ──────────────────────────────────────────────────────────
# "A" : use fixed sigma_rep prior
# "B" : estimate sigma_rep from structure function
USE_METHOD = "A"

# Prior value of sigma_rep (mm) — used by method A, and as initial guess for B
SIGMA_REP_PRIOR = 0.12   # mm

In [ ]:
# ── Best-estimate PWV ─────────────────────────────────────────────────────────
df_keep["PWV"]     = df_keep["PWV [mm]_ram"]
df_keep["PWV_err"] = df_keep["PWV [mm]_err_ram"]

# ── Time in minutes since first observation ───────────────────────────────────
t0_min = df_keep["MJD"].min() * 1440.0
df_keep["t_min"] = df_keep["MJD"] * 1440.0 - t0_min

# ── Night-by-night mean subtraction ──────────────────────────────────────────
night_means = df_keep.groupby("nightObs")["PWV"].transform("mean")
df_keep["PWV_night_mean"] = night_means
df_keep["dPWV"]           = df_keep["PWV"] - night_means

# ── Variance budget ───────────────────────────────────────────────────────────
pwv_var_obs  = df_keep["dPWV"].var(ddof=1)           # total observed
var_stat     = (df_keep["PWV_err"] ** 2).mean()       # photon noise (from PWV_err)
var_rep_A    = SIGMA_REP_PRIOR ** 2                   # repeatability prior
var_noise_A  = var_stat + var_rep_A                   # method A total noise
var_phys_A   = max(pwv_var_obs - var_noise_A, 1e-6)  # physical variance (method A)

n_nights = df_keep["nightObs"].nunique()
night_stats = df_keep.groupby("nightObs").agg(
    n_obs    = ("PWV", "count"),
    pwv_mean = ("PWV", "mean"),
    pwv_std  = ("PWV", "std"),
    dpwv_std = ("dPWV", "std"),
)

print(f"N observations              : {len(df_keep)}")
print(f"N nights (nightObs)         : {n_nights}")
print(f"Median obs per night        : {night_stats['n_obs'].median():.0f}")
print()
print(f"sigma_obs  (total intra-night)  : {np.sqrt(pwv_var_obs):.4f} mm")
print(f"sigma_stat (pooled PWV_err)     : {np.sqrt(var_stat):.4f} mm")
print(f"sigma_rep  (prior, method A)    : {SIGMA_REP_PRIOR:.4f} mm")
print(f"sigma_noise (method A)          : {np.sqrt(var_noise_A):.4f} mm")
print(f"sigma_phys  (method A)          : {np.sqrt(var_phys_A):.4f} mm")
print()
print(f"Uncorrected C(0) (PWV03)        ~ {1 - var_stat/pwv_var_obs:.3f}")
print(f"Corrected  C(0) (method A)      ~ {var_phys_A/(var_phys_A):.3f}  [= 1 by construction]")

display(night_stats.describe().round(3))

---
## 4. Pair computation — same-night pairs

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
LAG_MIN_MIN     =   1.0
LAG_MAX_MIN     = 600.0
N_BINS_LOG      =  50
N_MAX_SUBSAMPLE = 3000

rng = np.random.default_rng(seed=42)

nights = sorted(df_keep["nightObs"].unique())
print(f"Processing {len(nights)} nights...")

In [ ]:
# ── Accumulate (dt, product, sq_diff) across all nights ───────────────────────
# sq_diff = (δPWV_i - δPWV_j)^2  -> used for the structure function

dt_pairs_list      = []
prod_pairs_list    = []
sqdiff_pairs_list  = []

for night in nights:
    sub_n = df_keep[df_keep["nightObs"] == night]
    if len(sub_n) < 2:
        continue

    t_n  = sub_n["t_min"].values
    dp_n = sub_n["dPWV"].values

    if len(t_n) > N_MAX_SUBSAMPLE:
        idx_n = np.sort(rng.choice(len(t_n), N_MAX_SUBSAMPLE, replace=False))
        t_n   = t_n[idx_n]
        dp_n  = dp_n[idx_n]

    i_n, j_n = np.triu_indices(len(t_n), k=1)
    dt_n     = t_n[j_n] - t_n[i_n]
    prod_n   = dp_n[i_n] * dp_n[j_n]
    sq_n     = (dp_n[i_n] - dp_n[j_n]) ** 2

    ok_n = (dt_n >= LAG_MIN_MIN) & (dt_n <= LAG_MAX_MIN)
    dt_pairs_list.append(dt_n[ok_n])
    prod_pairs_list.append(prod_n[ok_n])
    sqdiff_pairs_list.append(sq_n[ok_n])

dt_pairs      = np.concatenate(dt_pairs_list)
prod_pairs    = np.concatenate(prod_pairs_list)
sqdiff_pairs  = np.concatenate(sqdiff_pairs_list)

print(f"Total same-night pairs within 10 h : {len(dt_pairs):,}")
if len(dt_pairs) > 0:
    print(f"Lag range (kept) : {dt_pairs.min():.2f} – {dt_pairs.max():.1f} min")

---
## 5. Method B — Structure Function & σ_rep estimation

We compute the **structure function** (SF) in the same log-bins as C:
$$
\mathrm{SF}(\Delta t_k) = \frac{1}{N_k} \sum_{\{i,j\}\in\mathrm{bin}_k}
  \bigl(\delta\mathrm{PWV}_i - \delta\mathrm{PWV}_j\bigr)^2.
$$
At zero physical lag all atmospheric signal cancels, leaving only
$\mathrm{SF}(0) = 2\,(\bar{\sigma}^2_{\rm stat} + \sigma^2_{\rm rep})$.
Fitting $\mathrm{SF}(\Delta t) = S_0 + A\,\Delta t^\beta$ and
extrapolating to $\Delta t=0$ gives $S_0$, from which:
$$
\sigma^2_{\rm rep,B} = \frac{S_0}{2} - \bar{\sigma}^2_{\rm stat}.
$$


In [ ]:
# ── Log bins (same grid used for C and SF) ────────────────────────────────────
bin_edges = np.logspace(
    np.log10(LAG_MIN_MIN),
    np.log10(LAG_MAX_MIN),
    N_BINS_LOG + 1
)
bin_centers_min = np.sqrt(bin_edges[:-1] * bin_edges[1:])

bin_idx = np.digitize(dt_pairs, bin_edges) - 1
valid   = (bin_idx >= 0) & (bin_idx < N_BINS_LOG)

# ── Binned structure function ─────────────────────────────────────────────────
sf      = np.full(N_BINS_LOG, np.nan)
sf_err  = np.full(N_BINS_LOG, np.nan)
n_pairs = np.zeros(N_BINS_LOG, dtype=int)

for k in range(N_BINS_LOG):
    mask = valid & (bin_idx == k)
    nk   = mask.sum()
    n_pairs[k] = nk
    if nk < 5:
        continue
    sq_k    = sqdiff_pairs[mask]
    sf[k]   = sq_k.mean()
    sf_err[k] = sq_k.std(ddof=1) / np.sqrt(nk)

lag_min = bin_centers_min
lag_hr  = bin_centers_min / 60.0

MIN_PAIRS = 5
good = n_pairs >= MIN_PAIRS

# ── Fit SF(Δt) = S0 + A*Δt^β over short lags (< 30 min) to extrapolate SF(0) ─
# Use only the first few short-lag bins where the power law is most evident.
FIT_MAX_MIN = 30.0   # use only bins below this lag for the SF extrapolation

fit_mask_sf = good & (lag_min <= FIT_MAX_MIN) & np.isfinite(sf) & np.isfinite(sf_err) & (sf_err > 0)

def sf_model(dt, S0, A, beta):
    return S0 + A * dt**beta

sf_fit_ok = False
if fit_mask_sf.sum() >= 4:
    try:
        x_sf = lag_min[fit_mask_sf]
        y_sf = sf[fit_mask_sf]
        e_sf = sf_err[fit_mask_sf]
        # initial guess: S0 ≈ first SF value, A small, beta~2/3
        p0 = [y_sf.min(), y_sf.ptp() / x_sf.max()**0.67, 0.67]
        popt_sf, pcov_sf = curve_fit(
            sf_model, x_sf, y_sf,
            p0=p0, sigma=e_sf, absolute_sigma=True,
            bounds=([0, 0, 0.1], [pwv_var_obs*2, np.inf, 2.0]),
            maxfev=20_000
        )
        perr_sf = np.sqrt(np.diag(pcov_sf))
        S0_fit, A_sf, beta_sf = popt_sf
        S0_err = perr_sf[0]
        sf_fit_ok = True

        var_noise_B  = S0_fit / 2.0   # = sigma^2_stat + sigma^2_rep
        var_rep_B    = max(var_noise_B - var_stat, 0.0)
        sigma_rep_B  = np.sqrt(var_rep_B)
        var_phys_B   = max(pwv_var_obs - var_noise_B, 1e-6)

        print("=== SF extrapolation fit  SF(Δt) = S0 + A*Δt^β ===")
        print(f"  S0   = {S0_fit:.5f} +/- {S0_err:.5f} mm^2")
        print(f"  beta = {beta_sf:.3f}")
        print(f"  => sigma_noise (method B) = {np.sqrt(var_noise_B):.4f} mm")
        print(f"  => sigma_rep   (method B) = {sigma_rep_B:.4f} mm")
        print(f"  => sigma_phys  (method B) = {np.sqrt(var_phys_B):.4f} mm")
    except Exception as exc:
        print(f"SF fit failed: {exc}")
else:
    print(f"Not enough bins below {FIT_MAX_MIN} min for SF fit ({fit_mask_sf.sum()} bins).")
    print("Falling back to method A.")

# ── Select active denominator based on USE_METHOD ─────────────────────────────
if USE_METHOD == "B" and sf_fit_ok:
    var_noise_active = var_noise_B
    sigma_rep_active = sigma_rep_B
    print("\nActive method: B (structure function)")
else:
    var_noise_active = var_noise_A
    sigma_rep_active = SIGMA_REP_PRIOR
    if USE_METHOD == "B" and not sf_fit_ok:
        print("\nFell back to method A (SF fit failed).")
    else:
        print("\nActive method: A (prior sigma_rep)")

pwv_var = max(pwv_var_obs - var_noise_active, 1e-6)  # denominator for C(Δt)

print(f"\n>>> Denominator sigma_phys = {np.sqrt(pwv_var):.4f} mm")
print(f">>> C(0) will be renormalised to 1.000 by construction")

In [ ]:
# ── Plot the structure function & fit ─────────────────────────────────────────
fig_sf, ax_sf = plt.subplots(figsize=(10, 5))

ax_sf.errorbar(
    lag_min[good], sf[good], yerr=sf_err[good],
    fmt='o', color='steelblue', ms=3.5, elinewidth=0.8, capsize=2,
    label=r'$\mathrm{SF}(\Delta t)$'
)

if sf_fit_ok:
    t_dense = np.logspace(np.log10(LAG_MIN_MIN), np.log10(FIT_MAX_MIN*2), 300)
    ax_sf.plot(t_dense, sf_model(t_dense, *popt_sf),
               color='darkred', lw=1.8, ls='--',
               label=rf'Fit: $S_0 + A\,\Delta t^{{\beta}}$, $S_0={S0_fit:.4f}$ mm$^2$')
    ax_sf.axhline(S0_fit, color='darkorange', lw=1.2, ls=':',
                  label=rf'$S_0=2\sigma^2_{{\rm noise}}$ = {S0_fit:.4f} mm$^2$')

ax_sf.axhline(2 * var_noise_active, color='green', lw=1.5, ls='-.',
              label=rf'$2\sigma^2_{{\rm noise}}$ (active) = {2*var_noise_active:.4f} mm$^2$')
ax_sf.axhline(2 * var_stat, color='gray', lw=1.0, ls=':',
              label=rf'$2\sigma^2_{{\rm stat}}$ = {2*var_stat:.4f} mm$^2$')

ax_sf.set_xscale('log')
ax_sf.set_xlim(LAG_MIN_MIN, LAG_MAX_MIN)
ax_sf.set_xlabel(r'$\Delta t$ [minutes]', fontsize=13)
ax_sf.set_ylabel(r'$\mathrm{SF}(\Delta t)$ [mm$^2$]', fontsize=13)
ax_sf.set_title('Structure function  —  extrapolation to Δt=0 gives σ²_noise', fontsize=11)
ax_sf.legend(fontsize=9)

fig_sf.tight_layout()
figfile_sf = f"{pathfigs}/{prefix}_{version_run}_structure_function{figtype}"
fig_sf.savefig(figfile_sf, bbox_inches='tight')
print(f"Saved: {figfile_sf}")
plt.show()

---
## 6. Corrected correlation function C(Δt)

The estimator is:
$$
\hat C_{\rm corr}(\Delta t_k) =
  \frac{\langle\delta\mathrm{PWV}_i\,\delta\mathrm{PWV}_j\rangle_{\Delta t_k}}
       {\sigma^2_{\rm phys}},
\quad
\sigma^2_{\rm phys} = \sigma^2_{\rm obs} - \sigma^2_{\rm noise}.
$$
Because $\sigma^2_{\rm phys}$ now accounts for both $\sigma_{\rm stat}$
and $\sigma_{\rm rep}$, the estimator satisfies
$\hat C_{\rm corr}(0) \approx 1$ without any subtraction in the numerator.


In [ ]:
# ── Corrected C(Δt) ───────────────────────────────────────────────────────────
corr     = np.full(N_BINS_LOG, np.nan)
corr_err = np.full(N_BINS_LOG, np.nan)

for k in range(N_BINS_LOG):
    mask = valid & (bin_idx == k)
    nk   = mask.sum()
    if nk < MIN_PAIRS:
        continue
    prods_k     = prod_pairs[mask]
    corr[k]     = prods_k.mean() / pwv_var          # corrected denominator
    corr_err[k] = prods_k.std(ddof=1) / (np.sqrt(nk) * pwv_var)

print(f"Bins with >= {MIN_PAIRS} pairs : {good.sum()} / {N_BINS_LOG}")
print(f"C at first bin (Δt ~ {lag_min[good][0]:.1f} min) = "
      f"{corr[good][0]:.3f}  (should be close to 1.0)")

---
## 7. Decorrelation timescale

In [ ]:
df_corr = pd.DataFrame({
    "lag_min" : lag_min[good],
    "lag_hr"  : lag_hr[good],
    "C"       : corr[good],
    "C_err"   : corr_err[good],
    "n_pairs" : n_pairs[good],
})

inv_e = 1.0 / np.e
cross = df_corr[df_corr["C"] < inv_e]
if len(cross) > 0:
    tau_min = cross.iloc[0]["lag_min"]
    tau_hr  = cross.iloc[0]["lag_hr"]
    print(f"Decorrelation timescale (C < 1/e) : {tau_min:.1f} min = {tau_hr:.2f} h")
else:
    tau_min = np.nan
    tau_hr  = np.nan
    print("C(Delta_t) does not drop below 1/e within 10 h")

display(df_corr)

---
## 8. Main figure — 2-panel (linear hours | log minutes)

In [ ]:
CORR_COLOR  = "steelblue"
ERR_COLOR   = "steelblue"
ZERO_COLOR  = "gray"
INV_E_COLOR = "darkorange"
TAU_COLOR   = "darkred"


def _decorate_corr_ax(ax, x_tau=None, xlabel="", xscale="linear",
                      xlim=None, ylim=(-0.5, 1.5)):
    ax.axhline(0.0,      ls="--", color=ZERO_COLOR,  lw=1.0, zorder=1)
    ax.axhline(1.0,      ls=":",  color="black",      lw=1.0, zorder=1, alpha=0.5)
    ax.axhline(1.0/np.e, ls=":",  color=INV_E_COLOR,  lw=1.5, zorder=1,
               label=r"$C = 1/e$")
    if x_tau is not None and np.isfinite(x_tau):
        unit = xlabel.split('[')[1].rstrip(']') if '[' in xlabel else ''
        ax.axvline(x_tau, ls="-", color=TAU_COLOR, lw=1.5, zorder=2,
                   label=rf"$\tau_{{1/e}} = {x_tau:.1f}$ {unit}")
    ax.set_ylabel(r"$C_{\rm corr}(\Delta t)$", fontsize=13)
    ax.set_xlabel(xlabel, fontsize=13)
    ax.set_xscale(xscale)
    if xlim is not None:
        ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.legend(fontsize=9, loc="upper right")


fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.subplots_adjust(wspace=0.32)

ax_lin = axes[0]
ax_lin.fill_between(
    lag_hr[good], corr[good] - corr_err[good], corr[good] + corr_err[good],
    alpha=0.25, color=ERR_COLOR
)
ax_lin.plot(lag_hr[good], corr[good],
            color=CORR_COLOR, lw=1.8, marker="o", ms=3.5, zorder=3,
            label=r"$C_{\rm corr}(\Delta t)$")
_decorate_corr_ax(
    ax_lin,
    x_tau  = tau_hr if np.isfinite(tau_hr) else None,
    xlabel = r"$\Delta t$ [hours]",
    xscale = "linear",
    xlim   = (0, LAG_MAX_MIN / 60.0),
)
ax_lin.set_title(f"Corrected correlation (method {USE_METHOD}) — linear hours", fontsize=11)

ax_log = axes[1]
ax_log.fill_between(
    lag_min[good], corr[good] - corr_err[good], corr[good] + corr_err[good],
    alpha=0.25, color=ERR_COLOR
)
ax_log.plot(lag_min[good], corr[good],
            color=CORR_COLOR, lw=1.8, marker="o", ms=3.5, zorder=3,
            label=r"$C_{\rm corr}(\Delta t)$")
_decorate_corr_ax(
    ax_log,
    x_tau  = tau_min if np.isfinite(tau_min) else None,
    xlabel = r"$\Delta t$ [minutes]",
    xscale = "log",
    xlim   = (LAG_MIN_MIN, LAG_MAX_MIN),
)
ax_log.set_title(f"Corrected correlation (method {USE_METHOD}) — log minutes", fontsize=11)

ax_log2 = ax_log.twiny()
ax_log2.set_xlim(np.array(ax_log.get_xlim()) / 60.0)
ax_log2.set_xscale("log")
ax_log2.set_xlabel(r"$\Delta t$ [hours]", fontsize=10, labelpad=6)

fig.suptitle(
    f"PWV corrected correlation (σ_rep={sigma_rep_active:.3f} mm, method {USE_METHOD}) — {version_run}",
    fontsize=11, y=1.02
)

figfile1 = f"{pathfigs}/{prefix}_{version_run}_2point_corr_corrected{figtype}"
fig.savefig(figfile1, bbox_inches="tight")
print(f"Saved: {figfile1}")
plt.show()

---
## 9. Zoom: sub-hour timescales

In [ ]:
ZOOM_MAX_MIN = 60.0
short = good & (lag_min <= ZOOM_MAX_MIN)

fig2, ax2 = plt.subplots(figsize=(10, 5))
ax2.fill_between(
    lag_min[short], corr[short] - corr_err[short], corr[short] + corr_err[short],
    alpha=0.25, color=ERR_COLOR
)
ax2.plot(lag_min[short], corr[short],
         color=CORR_COLOR, lw=2, marker="o", ms=4.5, zorder=3,
         label=r"$C_{\rm corr}(\Delta t)$")
_decorate_corr_ax(
    ax2,
    x_tau  = tau_min if (np.isfinite(tau_min) and tau_min <= ZOOM_MAX_MIN) else None,
    xlabel = r"$\Delta t$ [minutes]",
    xscale = "log",
    xlim   = (LAG_MIN_MIN, ZOOM_MAX_MIN),
)
ax2.set_title(r"Sub-hour corrected correlation — log minutes", fontsize=11)

ax2b = ax2.twiny()
ax2b.set_xlim(np.array(ax2.get_xlim()) / 60.0)
ax2b.set_xscale("log")
ax2b.set_xlabel(r"$\Delta t$ [hours]", fontsize=10, labelpad=6)

fig2.suptitle(
    f"PWV sub-hour corrected correlation (method {USE_METHOD}) — {version_run}",
    fontsize=12, y=1.02
)

figfile2 = f"{pathfigs}/{prefix}_{version_run}_2point_corr_subhour{figtype}"
fig2.savefig(figfile2, bbox_inches="tight")
print(f"Saved: {figfile2}")
plt.show()

---
## 10. Separate correlations per filter

Same correction applied per filter — σ_stat varies per filter (different
spectral bandpass → different photon statistics), but σ_rep is instrument-wide
and kept fixed.


In [ ]:
FILTERS_OF_INTEREST = ["empty", "OG550_65mm_1", "FELH0600"]
filter_color = {
    "empty"        : "#1f77b4",
    "OG550_65mm_1" : "#d62728",
    "FELH0600"     : "#2ca02c",
}
filter_label = {
    "empty"        : "empty",
    "OG550_65mm_1" : "OG550",
    "FELH0600"     : "FELH0600",
}

corr_by_filter     = {}
corr_err_by_filter = {}
npairs_by_filter   = {}

for filt in FILTERS_OF_INTEREST:
    sub_f = df_keep[df_keep["FILTER"] == filt]
    if len(sub_f) < 10:
        print(f"  {filt}: too few points ({len(sub_f)}), skipped")
        continue

    sub_f = sub_f.copy()
    night_means_f = sub_f.groupby("nightObs")["PWV"].transform("mean")
    sub_f["dPWV_f"] = sub_f["PWV"] - night_means_f

    # Per-filter variance budget:
    # sigma_rep is instrument-wide, sigma_stat is filter-specific
    var_obs_f    = sub_f["dPWV_f"].var(ddof=1)
    var_stat_f   = (sub_f["PWV_err"] ** 2).mean()
    var_noise_f  = var_stat_f + sigma_rep_active ** 2
    var_phys_f   = max(var_obs_f - var_noise_f, 1e-6)

    noise_frac_f = var_noise_f / var_obs_f
    print(f"  {filt}:")
    print(f"    sigma_obs={np.sqrt(var_obs_f):.3f} mm  "
          f"sigma_stat={np.sqrt(var_stat_f):.3f} mm  "
          f"sigma_rep={sigma_rep_active:.3f} mm  "
          f"sigma_phys={np.sqrt(var_phys_f):.3f} mm  "
          f"noise_frac={noise_frac_f:.3f}")

    dt_list_f, prod_list_f = [], []
    for night in sub_f["nightObs"].unique():
        sub_fn = sub_f[sub_f["nightObs"] == night]
        if len(sub_fn) < 2:
            continue
        t_fn  = sub_fn["t_min"].values
        dp_fn = sub_fn["dPWV_f"].values
        if len(t_fn) > N_MAX_SUBSAMPLE:
            idx_fn = np.sort(rng.choice(len(t_fn), N_MAX_SUBSAMPLE, replace=False))
            t_fn   = t_fn[idx_fn]
            dp_fn  = dp_fn[idx_fn]
        i_fn, j_fn = np.triu_indices(len(t_fn), k=1)
        dt_fn   = t_fn[j_fn] - t_fn[i_fn]
        prod_fn = dp_fn[i_fn] * dp_fn[j_fn]
        ok_fn   = (dt_fn >= LAG_MIN_MIN) & (dt_fn <= LAG_MAX_MIN)
        dt_list_f.append(dt_fn[ok_fn])
        prod_list_f.append(prod_fn[ok_fn])

    if not dt_list_f:
        continue
    dt_f   = np.concatenate(dt_list_f)
    prod_f = np.concatenate(prod_list_f)

    bidx_f = np.digitize(dt_f, bin_edges) - 1
    val_f  = (bidx_f >= 0) & (bidx_f < N_BINS_LOG)

    c_f  = np.full(N_BINS_LOG, np.nan)
    ce_f = np.full(N_BINS_LOG, np.nan)
    np_f = np.zeros(N_BINS_LOG, dtype=int)

    for k in range(N_BINS_LOG):
        m  = val_f & (bidx_f == k)
        nk = m.sum()
        np_f[k] = nk
        if nk < MIN_PAIRS:
            continue
        pk = prod_f[m]
        c_f[k]  = pk.mean() / var_phys_f
        ce_f[k] = pk.std(ddof=1) / (np.sqrt(nk) * var_phys_f)

    corr_by_filter[filt]     = c_f
    corr_err_by_filter[filt] = ce_f
    npairs_by_filter[filt]   = np_f
    print(f"    {len(sub_f)} obs, {len(dt_f):,} same-night pairs")

print("Done.")

In [ ]:
fig4, axes4 = plt.subplots(1, 2, figsize=(16, 6))
fig4.subplots_adjust(wspace=0.32)

for ax4, xdata, xlabel, xscale, xlim in [
    (axes4[0], lag_hr,  r"$\Delta t$ [hours]",  "linear",
     (0, LAG_MAX_MIN / 60.0)),
    (axes4[1], lag_min, r"$\Delta t$ [minutes]", "log",
     (LAG_MIN_MIN, LAG_MAX_MIN)),
]:
    ax4.plot(xdata[good], corr[good],
             color="black", lw=1.0, alpha=0.4, ls="--", label="All filters")
    for filt in FILTERS_OF_INTEREST:
        if filt not in corr_by_filter:
            continue
        c_f  = corr_by_filter[filt]
        ce_f = corr_err_by_filter[filt]
        gf   = npairs_by_filter[filt] >= MIN_PAIRS
        ax4.fill_between(
            xdata[gf], c_f[gf] - ce_f[gf], c_f[gf] + ce_f[gf],
            alpha=0.15, color=filter_color[filt]
        )
        ax4.plot(xdata[gf], c_f[gf],
                 color=filter_color[filt], lw=1.8, marker="o", ms=3,
                 label=filter_label[filt])
    ax4.axhline(0.0,      ls="--", color=ZERO_COLOR,  lw=1.0)
    ax4.axhline(1.0,      ls=":",  color="black",      lw=0.8, alpha=0.5)
    ax4.axhline(1.0/np.e, ls=":",  color=INV_E_COLOR,  lw=1.5,
                label=r"$C = 1/e$")
    ax4.set_xscale(xscale)
    ax4.set_xlim(xlim)
    ax4.set_ylim(-1.35, 1.35)
    ax4.set_xlabel(xlabel, fontsize=13)
    ax4.set_ylabel(r"$C_{\rm corr}(\Delta t)$", fontsize=13)
    ax4.legend(fontsize=9, loc="upper right")

axes4[0].set_title("Per-filter corrected correlation — linear hours", fontsize=11)
axes4[1].set_title("Per-filter corrected correlation — log minutes",  fontsize=11)
fig4.suptitle(
    f"Per-filter corrected correlation (σ_rep={sigma_rep_active:.3f} mm, method {USE_METHOD}) — {version_run}",
    fontsize=12, y=1.01
)
figfile4 = f"{pathfigs}/{prefix}_{version_run}_2point_corr_per_filter{figtype}"
fig4.savefig(figfile4, bbox_inches="tight")
print(f"Saved: {figfile4}")
plt.show()

---
## 11. Exponential fit

In [ ]:
def exp_decay(dt, A, tau):
    return A * np.exp(-dt / tau)


fit_mask = (good
            & (lag_min <= LAG_MAX_MIN)
            & (corr > 0)
            & np.isfinite(corr)
            & np.isfinite(corr_err)
            & (corr_err > 0))

x_fit = lag_min[fit_mask]
y_fit = corr[fit_mask]
e_fit = corr_err[fit_mask]

tau_guess = tau_min if np.isfinite(tau_min) else 60.0

fit_ok = False
try:
    popt, pcov = curve_fit(
        exp_decay, x_fit, y_fit,
        p0=[1.0, tau_guess],
        sigma=e_fit, absolute_sigma=True,
        bounds=([0, 0.1], [2.0, LAG_MAX_MIN]),
        maxfev=20_000
    )
    perr = np.sqrt(np.diag(pcov))
    A_fit, tau_fit_min = popt
    A_err, tau_fit_err = perr
    tau_fit_hr = tau_fit_min / 60.0
    print("=== Exponential fit  C(dt) = A * exp(-dt/tau) ===")
    print(f"  A   = {A_fit:.4f} +/- {A_err:.4f}")
    print(f"  tau = {tau_fit_min:.1f} +/- {tau_fit_err:.1f} min")
    print(f"      = {tau_fit_hr:.2f} +/- {tau_fit_err/60:.2f} h")
    fit_ok = True
except Exception as exc:
    print(f"Fit failed: {exc}")

In [ ]:
fig5, axes5 = plt.subplots(1, 2, figsize=(16, 6))
fig5.subplots_adjust(wspace=0.32)

if fit_ok:
    t_dense_min = np.logspace(np.log10(LAG_MIN_MIN), np.log10(LAG_MAX_MIN), 500)
    t_dense_hr  = t_dense_min / 60.0
    c_dense     = exp_decay(t_dense_min, *popt)

for ax5, xdata, x_dense, xlabel, xscale, xlim, x_tau in [
    (axes5[0], lag_hr,  t_dense_hr  if fit_ok else None,
     r"$\Delta t$ [hours]",   "linear",
     (0, LAG_MAX_MIN / 60.0),
     tau_fit_hr  if fit_ok else np.nan),
    (axes5[1], lag_min, t_dense_min if fit_ok else None,
     r"$\Delta t$ [minutes]", "log",
     (LAG_MIN_MIN, LAG_MAX_MIN),
     tau_fit_min if fit_ok else np.nan),
]:
    ax5.fill_between(
        xdata[good], corr[good] - corr_err[good], corr[good] + corr_err[good],
        alpha=0.25, color=CORR_COLOR
    )
    ax5.plot(xdata[good], corr[good],
             color=CORR_COLOR, lw=1.5, marker="o", ms=3.5, zorder=3,
             label=r"$C_{\rm corr}(\Delta t)$")
    if fit_ok:
        ax5.plot(x_dense, c_dense,
                 color="darkred", lw=2.0, ls="-", zorder=5,
                 label=(rf"$A\,e^{{-\Delta t/\tau}}$"
                        rf", $A={A_fit:.3f}$, $\tau={tau_fit_hr:.1f}$ h"))
        unit = xlabel.split('[')[1].rstrip(']') if '[' in xlabel else ''
        ax5.axvline(x_tau, ls="--", color=TAU_COLOR, lw=1.5,
                    label=rf"$\tau = {x_tau:.1f}$ {unit}")
    ax5.axhline(0.0,      ls="--", color=ZERO_COLOR,  lw=1.0)
    ax5.axhline(1.0,      ls=":",  color="black",      lw=0.8, alpha=0.5)
    ax5.axhline(1.0/np.e, ls=":",  color=INV_E_COLOR,  lw=1.5,
                label=r"$C = 1/e$")
    ax5.set_xscale(xscale)
    ax5.set_xlim(xlim)
    ax5.set_ylim(-1.5, 1.5)
    ax5.set_xlabel(xlabel, fontsize=13)
    ax5.set_ylabel(r"$C_{\rm corr}(\Delta t)$", fontsize=13)
    ax5.legend(fontsize=9, loc="upper right")

axes5[0].set_title("Exponential fit — linear hours", fontsize=11)
axes5[1].set_title("Exponential fit — log minutes",  fontsize=11)
fig5.suptitle(
    f"Corrected correlation + exp fit (method {USE_METHOD}, σ_rep={sigma_rep_active:.3f} mm) — {version_run}",
    fontsize=12, y=1.01
)
figfile5 = f"{pathfigs}/{prefix}_{version_run}_2point_corr_expfit{figtype}"
fig5.savefig(figfile5, bbox_inches="tight")
print(f"Saved: {figfile5}")
plt.show()

---
## 12. Summary & export

In [ ]:
print("=== PWV corrected two-point correlation — summary ===")
print(f"  Method                      : {USE_METHOD}")
print(f"  N observations (cuts)       : {len(df_keep)}")
print(f"  N nights                    : {n_nights}")
print(f"  Same-night pairs (10 h)     : {len(dt_pairs):,}")
print()
print(f"  sigma_obs  (total)          : {np.sqrt(pwv_var_obs):.4f} mm")
print(f"  sigma_stat (PWV_err)        : {np.sqrt(var_stat):.4f} mm")
print(f"  sigma_rep  (active)         : {sigma_rep_active:.4f} mm")
print(f"  sigma_noise (active)        : {np.sqrt(var_noise_active):.4f} mm")
print(f"  sigma_phys  (denominator)   : {np.sqrt(pwv_var):.4f} mm")
print()
if np.isfinite(tau_min):
    print(f"  1/e crossing (empirical)    : {tau_min:.1f} min = {tau_hr:.2f} h")
else:
    print("  C(Delta_t) does not drop below 1/e within 10 h")
if fit_ok:
    print(f"  Exp-fit A                   : {A_fit:.4f} +/- {A_err:.4f}")
    print(f"  Exp-fit tau                 : {tau_fit_min:.1f} +/- {tau_fit_err:.1f} min")
    print(f"                              = {tau_fit_hr:.2f} +/- {tau_fit_err/60:.2f} h")

In [ ]:
df_corr_out = pd.DataFrame({
    "lag_min" : lag_min,
    "lag_hr"  : lag_hr,
    "C"       : corr,
    "C_err"   : corr_err,
    "n_pairs" : n_pairs,
    "SF"      : sf,
    "SF_err"  : sf_err,
})
csv_file = f"{pathfigs}/{prefix}_{version_run}_2point_corr_repcorr.csv"
df_corr_out.to_csv(csv_file, index=False, float_format="%.6f")
print(f"Saved: {csv_file}")

results = {
    "version_run"           : version_run,
    "correction_method"     : USE_METHOD,
    "sigma_rep_prior"       : SIGMA_REP_PRIOR,
    "sigma_rep_active"      : float(sigma_rep_active),
    "N_obs"                 : int(len(df_keep)),
    "N_nights"              : int(n_nights),
    "N_pairs_same_night_10h": int(len(dt_pairs)),
    "pwv_sigma_obs"         : float(np.sqrt(pwv_var_obs)),
    "pwv_sigma_stat"        : float(np.sqrt(var_stat)),
    "pwv_sigma_noise"       : float(np.sqrt(var_noise_active)),
    "pwv_sigma_phys"        : float(np.sqrt(pwv_var)),
    "tau_1e_min"            : float(tau_min) if np.isfinite(tau_min) else None,
    "tau_1e_hr"             : float(tau_hr)  if np.isfinite(tau_hr)  else None,
}
if sf_fit_ok:
    results["SF0_fit"]        = float(S0_fit)
    results["SF0_err"]        = float(S0_err)
    results["sigma_rep_B"]    = float(sigma_rep_B)
if fit_ok:
    results.update({
        "exp_A"          : float(A_fit),
        "exp_A_err"      : float(A_err),
        "exp_tau_min"    : float(tau_fit_min),
        "exp_tau_err_min": float(tau_fit_err),
        "exp_tau_hr"     : float(tau_fit_hr),
    })

json_file = f"{pathfigs}/{prefix}_{version_run}_2point_corr_repcorr_results.json"
with open(json_file, "w") as fh:
    json.dump(results, fh, indent=2)
print(f"Saved: {json_file}")
print(json.dumps(results, indent=2))